# Clasificación de madurez del proyecto: Demo/Prototipo vs. Release

Este notebook implementa un flujo completo de Machine Learning para detectar si un nombre de proyecto de videojuego corresponde a una entrega temprana (`demo`, `prototype`, `alpha`, `beta`, `test`, `fus`) o a una versión final/complete release. Se emplea ETL sobre texto no estructurado, extracción de atributos, TF-IDF sobre n-gramas de caracteres y un clasificador supervisado.

In [ ]:
import re
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix

print('Librerías cargadas correctamente.')


In [ ]:
# ETAPA 1: ETL Y PARSING DE DATOS

# Ejemplo de texto no estructurado (string dump) con nombres de proyectos, versiones y estados
raw_text = '''
project_name;version;notes
crash-demo-v1.0;1.0;demo build for testing
castle-quest;1.2;full release
alpha-racer-proto;0.9;prototype alpha test
beta-town;0.8;beta version
fusion-ride-fus;2.4;fus release candidate
space-wars;3.0;complete game
test-drive-2;2.1;test build
monster-quest-demo;1.0;demo release
final-escape-v1.5;1.5;complete release
proto-landing;0.4;proto concept

# Ruido / basura añadida para simular texto no estructurado
\x00\x01\x02demo\x1falpha\x00\x7fPROTO\x00
'''

# Filtro de ruido binario / ASCII no deseado
cleaned_text = re.sub(r'[^\x20-\x7E\n\r\t]', ' ', raw_text)
cleaned_text = re.sub(r'\s+', ' ', cleaned_text).strip()

# Extraer slugs y tokens útiles

# Se considera que la información útil puede venir como texto con guiones, puntos, 'v', 'proto', 'demo', etc.
# Para un caso real, esto se puede adaptar a la salida de la base de datos.
rows = []
for line in cleaned_text.split('\n'):
    line = line.strip()
    if not line or line.startswith('project_name') or line.startswith('#'):
        continue
    parts = [p.strip() for p in re.split(r'[;|\t]+', line) if p.strip()]
    if len(parts) >= 3:
        project = parts[0]
        version = parts[1]
        notes = ' '.join(parts[2:])
    elif len(parts) == 2:
        project = parts[0]
        version = parts[1]
        notes = ''
    else:
        continue

    slug = re.sub(r'[^a-zA-Z0-9\-\s]', ' ', project.lower())
    slug = re.sub(r'\s+', '-', slug.strip())
    slug = re.sub(r'-+', '-', slug)
    slug = slug.strip('-')

    tokens = [t for t in re.split(r'[^a-zA-Z0-9]+', slug) if t]
    rows.append({
        'project_name': project,
        'version': version,
        'notes': notes,
        'slug': slug,
        'tokens': tokens,
    })

# Crear DataFrame y etiqueta objetivo basada en palabras clave de madurez temprana
# 1 = demo/prototipo, 0 = final/release
keywords_demo = ['demo', 'fus', 'test', 'alpha', 'beta', 'proto', 'prototype']

df = pd.DataFrame(rows)
df['slug_clean'] = df['slug'].fillna('')
df['es_demo_prototipo'] = df['slug_clean'].str.lower().str.contains('|'.join(keywords_demo), regex=True).astype(int)

print('DataFrame generado:')
df.head()
print('\nInfo del DataFrame:')
df.info()


In [ ]:
# ETAPA 2: PROCESAMIENTO DE LENGUAJE NATURAL (NLP)

# Métricas numéricas del nombre / slug
# - longitud del slug
# - número de palabras separadas por guiones
# - presencia de números de versión
# - presencia de sufijos asociados a estado temprano

def extract_numeric_features(slug):
    slug = str(slug).lower()
    words = [w for w in slug.split('-') if w]
    has_version_number = 1 if re.search(r'\d+(?:\.\d+)?', slug) else 0
    has_demo_marker = 1 if re.search(r'(demo|fus|test|alpha|beta|proto|prototype)', slug) else 0
    return {
        'slug_length': len(slug),
        'num_words': len(words),
        'has_version_number': has_version_number,
        'has_demo_marker': has_demo_marker,
    }

features = df['slug_clean'].apply(extract_numeric_features).apply(pd.Series)
df = pd.concat([df, features], axis=1)

# Vectorización TF-IDF sobre subcadenas de caracteres (n-grams)
tfidf = TfidfVectorizer(
    analyzer='char_wb',
    ngram_range=(3, 5),
    lowercase=True,
    min_df=1
)

X_tfidf = tfidf.fit_transform(df['slug_clean'])
X_numeric = df[['slug_length', 'num_words', 'has_version_number', 'has_demo_marker']].values

from scipy.sparse import hstack
X = hstack([X_tfidf, np.asarray(X_numeric) * 1])

# Variable objetivo
Y = df['es_demo_prototipo']

print('Forma final de X:', X.shape)
print('Distribución de la variable objetivo:')
print(Y.value_counts(normalize=True))


In [ ]:
# ETAPA 3: ENTRENAMIENTO DEL MODELO

X_train, X_test, y_train, y_test = train_test_split(
    X,
    Y,
    test_size=0.3,
    random_state=42,
    stratify=Y
)

# Modelo de clasificación elegido: Regresión Logística con features TF-IDF + métricas numéricas
clf = LogisticRegression(max_iter=2000, random_state=42)
clf.fit(X_train, y_train)

y_pred = clf.predict(X_test)

print('Entrenamiento finalizado correctamente.')
print('Número de muestras de entrenamiento:', X_train.shape[0])
print('Número de muestras de prueba:', X_test.shape[0])


In [ ]:
# ETAPA 4: EVALUACIÓN Y VISUALIZACIÓN

# Exactitud global
accuracy = accuracy_score(y_test, y_pred)
print('Accuracy Score:', round(accuracy, 4))

# Informe detallado de clasificación
print('\nClassification Report:\n')
print(classification_report(y_test, y_pred, target_names=['Juego Completo', 'Demo/Prototipo']))

# Matriz de confusión
cm = confusion_matrix(y_test, y_pred)
plt.figure(figsize=(7, 5))
sns.heatmap(
    cm,
    annot=True,
    fmt='d',
    cmap='Blues',
    xticklabels=['Juego Completo', 'Demo/Prototipo'],
    yticklabels=['Juego Completo', 'Demo/Prototipo']
)
plt.title('Matriz de Confusión - Clasificación de madurez del proyecto')
plt.xlabel('Predicción')
plt.ylabel('Valor real')
plt.tight_layout()
plt.show()


## Conclusión

Clasificar si un proyecto es una demo o un prototipo frente a una versión final no es una tarea aleatoria ni una simple regla manual: es un problema de Machine Learning bien definido porque la nomenclatura, los sufijos de versión y los identificadores semánticos contienen patrones repetibles que pueden aprenderse automáticamente. En la práctica, palabras como `demo`, `alpha`, `beta`, `proto`, `test` o `fus` aparecen con mucha frecuencia en entregas tempranas, mientras que títulos más limpios o sin estos indicadores suelen corresponder a releases finales. Gracias a los métodos de NLP y clasificación supervisada, el modelo puede detectar esos patrones de forma sistemática, generalizar a nuevos nombres y ofrecer una decisión precisa y reproducible, algo mucho más robusto que depender únicamente de observación humana.